In [1]:
# Task 1.1 Read appointment.csv using only csv and json

In [2]:
# Import tools
import csv
import json

In [3]:
# File location and reading
file_path = "/content/appointments.csv"
with open(file_path, newline = "") as file:
  reader = csv.DictReader(file) # reads each line as a dictionary
  header = reader.fieldnames # the list of column names
  all_rows = list(reader) # turn all rows into one list so we can loop,

In [4]:
print("Header :", header)
print("Row Count", len(all_rows)) # counts items in the list

Header : ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row Count 1020


In [5]:
# function for parsing

def parse_row(row):
  converted = dict(row) # make a copy so the original stays as is
  converted["age"] = int(row["age"])
  converted["fee_inr"] = int(row["fee_inr"]) #int coversion
  converted ["prior_no_shows"] = int(row["prior_no_shows"])
  converted ["distance_km"] = float(row["distance_km"]) # float conversion

  return converted # hand the dictionary to the function caller

In [6]:
#Loop over rows and sort as accepted or rejected

accepted = []
rejected = [] # Two empty lists like baskets

# for visits every row one at a time
for row in all_rows:
  try:
    good_row = parse_row(row)
    accepted.append(good_row)# succeeded
  except ValueError:
    rejected.append(row) # failed (raw row)



In [7]:
# Check the output
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("TOTAL:", len(accepted) + len(rejected)) # should be equal to row count
#

Accepted rows: 988
Rejected rows: 32
TOTAL: 1020


In [8]:
# rejected rows as json file
with open ("rejected_rows.json","w") as out_file: # w means write - create or overwrite
  json.dump(rejected, out_file, indent = 2) # 2 for better readability
print("Saved rejected_rows.json")

Saved rejected_rows.json


In [9]:
# Section 1.2- Inspect, Clean and Pass 1
import pandas as pd
df = pd.read_csv("/content/appointments.csv")# read the file
print("Shape", df.shape) # read all contents, sizes ...
df.info()
print("Missing values per column:")
print(df.isnull().sum())
print("Duplicated Rows:", df.duplicated().sum())

Shape (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
Missing values per column:
appointment_id      0
patient_id          0
clin

In [10]:
#Drop duplicate rows
df = df.drop_duplicates()
print("Rows after duplicate removal:", len(df))
#clean booking channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after cleaning booking_channel:",len(df))
#convert date columns
df["appt_date"] = pd.to_datetime(df["appt_date"], errors ="coerce")
#convert booked_on, with out coerce(as per brief)
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Rows after conv booked_on:",len(df))
#drop rows with missing values
missing_rows = df["appt_date"].isnull() | df["status"].isnull()
df = df[~missing_rows]
print("After dropping missing_rows removal:",len(df))
#


Rows after duplicate removal: 1000
Rows after cleaning booking_channel: 1000
Rows after conv booked_on: 1000
After dropping missing_rows removal: 990


/tmp/ipykernel_6886/1549471677.py:8: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["appt_date"] = pd.to_datetime(df["appt_date"], errors ="coerce")
/tmp/ipykernel_6886/1549471677.py:10: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["booked_on"] = pd.to_datetime(df["booked_on"])


In [11]:
# Connect Google Drive so files persist between sessions
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
# Section 1.3 - Clean, pass 2: validity, gaps and outliers
import numpy as np
invalid_age = (df["age"] < 1) | (df["age"] > 100) #clean age columns
print("Invalid ages set to missing" , invalid_age.sum())

df.loc[invalid_age, "age"] = np.nan # value sets to cells with NaN
print("Step (a) - Missing age now:", df["age"].isnull().sum())

age_median = df["age"].median() # median value for NaN cells
distance_median = df["distance_km"].median() #""
print("Age median:", age_median)
print("Distance median:", distance_median)

#NaN to median and rounded as integer
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)
print("Step (b) - Missing age after fill:", df["age"].isnull().sum())
print("Step (b) - Missing distance_km after fill:", df["distance_km"].isnull().sum())


# capping outliers in diatance and fee
# we repeat for each column and hence loop
# Empty dictionaries to store results, so we can print them after the loop.
stats = {}
capped_counts = {}
for column in["distance_km","fee_inr"]:

  q1 = df[column].quantile(0.25)
  q3 = df[column].quantile(0.75) # 25th and 75the percentile for IQR
  iqr = q3 - q1
  lower_bound = q1 - 1.5 * iqr
  upper_bound = q3 + 1.5 * iqr # outside the bounadries are outliers

  capped_counts[column] =((df[column] < lower_bound) | (df[column] > upper_bound)).sum()

  # Save this column's statistics.
  stats[column] = {"Q1": q1, "Q3": q3, "IQR": iqr,
                     "Lower fence": lower_bound, "Upper fence": upper_bound}
  df[column] = np.clip(df[column], lower_bound, upper_bound)

  # Print after the loop has finished.
print("\nStep (c) - Statistical parameters:")
for column in stats:
    print(column, ":", stats[column])

print("\nStep (c) - Values capped:")
for column in capped_counts:
    print("Capped in", column, ":", capped_counts[column])

# pull the values outside the bound to bound value
df[column] = np.clip(df[column],lower_bound, upper_bound)

#round the columns

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("\nStep (d) - Data types after rounding:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

print("\nStep (d) - Summary of the three columns:")
print(df[["age", "distance_km", "fee_inr"]].describe())

print("\nFinal row count:", len(df))


Invalid ages set to missing 5
Step (a) - Missing age now: 17
Age median: 48.0
Distance median: 8.25
Step (b) - Missing age after fill: 0
Step (b) - Missing distance_km after fill: 0

Step (c) - Statistical parameters:
distance_km : {'Q1': np.float64(4.625), 'Q3': np.float64(13.0), 'IQR': np.float64(8.375), 'Lower fence': np.float64(-7.9375), 'Upper fence': np.float64(25.5625)}
fee_inr : {'Q1': np.float64(500.0), 'Q3': np.float64(787.5), 'IQR': np.float64(287.5), 'Lower fence': np.float64(68.75), 'Upper fence': np.float64(1218.75)}

Step (c) - Values capped:
Capped in distance_km : 26
Capped in fee_inr : 3

Step (d) - Data types after rounding:
age              int64
distance_km    float64
fee_inr          int64
dtype: object

Step (d) - Summary of the three columns:
              age  distance_km      fee_inr
count  990.000000   990.000000   990.000000
mean    48.173737     9.500909   635.714141
std     16.655376     6.160314   183.802489
min     18.000000     0.600000   300.000000
25%

In [13]:
# Task 1.4 - Festures and Numpy Summary

In [14]:

#  Create the features
# DATA SOURCE: df (cleaned table from Tasks 1.2 and 1.3, 990 rows

# lead_days = number of days between booking and appointment.
# Subtracting two dates gives a time difference;
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# no_show = 1 if status is "No-Show", otherwise 0.
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("SECTION 1 - Features created")
print(df[["lead_days", "no_show"]].head())


#  Build the NumPy array X

# to_numpy() turns the four columns into a NumPy array.
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

print("\nSECTION 2 - Shape of X:", X.shape)


#  (a) Mean of each column

# axis=0 means "go down the rows", giving one mean per column.
import numpy as np
column_means = np.mean(X, axis=0)

print("\nSECTION 3 (a) - Column means")
print("Columns:", ["age", "distance_km", "fee_inr", "lead_days"])
print("Means:  ", column_means)


#  (b) Rows with lead_days > 14

# X[:, 3] is the lead_days column.
# The mask is True where lead_days is above 14, otherwise False.
mask = X[:, 3] > 14

print("\nSECTION 4 (b) - lead_days > 14")
print("Number of rows:", mask.sum())
print("Share of rows (%):", round(mask.mean() * 100, 1))


# (c) z-scores of distance_km

# X[:, 1] is the distance_km column.
x = X[:, 1]

# z-score = (value - mean) / standard deviation
z = (x - np.mean(x)) / np.std(x)

# Count rows where the z-score is more than 2 away from zero (either side).
count_outliers = (np.abs(z) > 2).sum()

print("\nSECTION 5 (c) - distance_km z-scores")
print("Rows with |z| > 2:", count_outliers)

SECTION 1 - Features created
   lead_days  no_show
0          1        1
1         10        0
2          6        0
4          4        0
5         40        1

SECTION 2 - Shape of X: (990, 4)

SECTION 3 (a) - Column means
Columns: ['age', 'distance_km', 'fee_inr', 'lead_days']
Means:   [ 48.17373737   9.50090909 635.71414141  11.81919192]

SECTION 4 (b) - lead_days > 14
Number of rows: 296
Share of rows (%): 29.9

SECTION 5 (c) - distance_km z-scores
Rows with |z| > 2: 53


In [15]:
# Enrich the clinics from a live API

# SECTION 1: Import the tools

# requests lets Python call a web API.
import requests
import pandas as pd

# SECTION 2: Load clinics.csv

clinics = pd.read_csv("/content/clinics.csv")

print("SECTION 2 - Clinics loaded")
print("Shape:", clinics.shape)
print("Columns:", list(clinics.columns))



# SECTION 3: Function that gets the temperature for ONE clinic

# It returns two things: the temperature (or None) and a reason text.
def get_temp(latitude, longitude):

    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": latitude, "longitude": longitude,
              "hourly": "temperature_2m", "forecast_days": 1}

    reply = None
    reason = ""

    # range(2) = two attempts: the first try plus one retry.
    for attempt in range(2):
        try:
            reply = requests.get(url, params=params, timeout=10)
            break                                  # success, stop retrying
        except requests.exceptions.RequestException as error:
            reason = "request failed: " + str(error)
            reply = None

    # Both attempts failed with an exception.
    if reply is None:
        return None, reason

    # The API answered but not with status 200.
    if reply.status_code != 200:
        return None, "status code was " + str(reply.status_code)

    # Read the JSON and check that "hourly" exists.
    data = reply.json()
    if "hourly" not in data:
        return None, "hourly is missing in the reply"

    # First value = the midnight-UTC reading.
    return data["hourly"]["temperature_2m"][0], ""


# SECTION 4: Call the API for each of the 8 clinics

temps = []     # will hold one temperature (or None) per clinic

print("\nSECTION 4 - API calls")
for index, row in clinics.iterrows():

    temp, reason = get_temp(row["latitude"], row["longitude"])
    temps.append(temp)

    if temp is None:
        print(row["clinic_name"], "-> FAILED:", reason)
    else:
        print(row["clinic_name"], "-> temp_c:", temp)


# SECTION 5: Add temp_c and save clinics_enriched.csv

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)

print("\nSECTION 5 - Saved clinics_enriched.csv")
print(clinics)

SECTION 2 - Clinics loaded
Shape: (8, 6)
Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

SECTION 4 - API calls
Kothrud Family Clinic -> temp_c: 17.1
Hadapsar Care Centre -> temp_c: 18.2
Sitabuldi Health Point -> temp_c: 20.7
Malviya Nagar Clinic -> temp_c: 17.1
Vaishali Nagar Clinic -> temp_c: 17.1
Edappally Medical Centre -> temp_c: 24.7
Vijay Nagar Clinic -> temp_c: 20.3
Gomti Nagar Care Point -> temp_c: 15.7

SECTION 5 - Saved clinics_enriched.csv
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre  

In [16]:

#  Save appointments_clean.csv and write load_data.sql

# Sort by appointment_id

df = df.sort_values("appointment_id")
print("SECTION 1 - Sorted by appointment_id")
print(df["appointment_id"].head())

#  Format both date columns as YYYY-MM-DD text

df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

print("\nSECTION 2 - Dates as text")
print(df[["booked_on", "appt_date"]].head())


#  Keep exactly the 16 columns, in the required order

columns_16 = ["appointment_id", "patient_id", "clinic_id", "booked_on",
              "appt_date", "age", "gender", "distance_km", "fee_inr",
              "booking_channel", "visit_type", "reminder_sent",
              "prior_no_shows", "status", "lead_days", "no_show"]

df = df[columns_16]

print("\nSECTION 3 - Columns:", list(df.columns))
print("Number of columns:", len(df.columns))

# Checks before exporting

print("\nSECTION 4 - Checks")
print("appointment_id is unique:", df["appointment_id"].is_unique)
print("Missing values per column:")
print(df.isnull().sum())
print("Total missing values:", df.isnull().sum().sum())

# Save appointments_clean.csv

df.to_csv("appointments_clean.csv", index=False)

print("\nSECTION 5 - Saved appointments_clean.csv with", len(df), "rows")

#  Build the clinics INSERT statements

# Read the clinics file that Task 1.5 saved.
clinics = pd.read_csv("clinics_enriched.csv")

# This list will hold every line of the SQL file, in order.
sql_lines = []

for index, row in clinics.iterrows():

    # A missing temperature is written as NULL (no quotes).
    if pd.isnull(row["temp_c"]):
        temp = "NULL"
    else:
        temp = row["temp_c"]

    line = f"INSERT INTO clinics VALUES ('{row['clinic_id']}','{row['clinic_name']}','{row['city']}','{row['tier']}',{row['latitude']},{row['longitude']},{temp});"
    sql_lines.append(line)

print("\nSECTION 6 - Clinics statements:", len(sql_lines))
print(sql_lines[0])

#  Build the appointments INSERT statements

# Clinics come first (above), because appointments point to them.
appointment_lines = []

for index, row in df.iterrows():

    line = f"INSERT INTO appointments VALUES ('{row['appointment_id']}','{row['patient_id']}','{row['clinic_id']}','{row['booked_on']}','{row['appt_date']}',{row['age']},'{row['gender']}',{row['distance_km']},{row['fee_inr']},'{row['booking_channel']}','{row['visit_type']}','{row['reminder_sent']}',{row['prior_no_shows']},'{row['status']}',{row['lead_days']},{row['no_show']});"
    appointment_lines.append(line)

sql_lines = sql_lines + appointment_lines

print("\nSECTION 7 - Appointments statements:", len(appointment_lines))
print(appointment_lines[0])

# SECTION 8: Write load_data.sql

with open("load_data.sql", "w") as sql_file:
    for line in sql_lines:
        sql_file.write(line + "\n")

print("\nSECTION 8 - Saved load_data.sql with", len(sql_lines), "statements")

SECTION 1 - Sorted by appointment_id
0    A0001
1    A0002
2    A0003
4    A0004
5    A0005
Name: appointment_id, dtype: object

SECTION 2 - Dates as text
    booked_on   appt_date
0  2025-01-24  2025-01-25
1  2025-05-27  2025-06-06
2  2025-03-24  2025-03-30
4  2025-04-18  2025-04-22
5  2024-11-23  2025-01-02

SECTION 3 - Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']
Number of columns: 16

SECTION 4 - Checks
appointment_id is unique: True
Missing values per column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
T

In [17]:
print(df["no_show"].sum()) # checking sql output for crossverification

218


In [18]:
# ============================================================
#2.2 - Reproduce clinic counts and no-show counts in Python
# ============================================================

# ------------------------------------------------------------
# Group by clinic_id and count
# ------------------------------------------------------------
# count = number of appointments, sum = number of no-shows (no_show is 0/1).
summary = df.groupby("clinic_id")["no_show"].agg(appointments="count", no_shows="sum").reset_index()

# ------------------------------------------------------------
# SECTION 2: Add the clinic names so the rows match the SQL output
# ------------------------------------------------------------
summary = summary.merge(clinics[["clinic_id", "clinic_name"]], on="clinic_id")

# ------------------------------------------------------------
# SECTION 3: Print the result
# ------------------------------------------------------------
print("Clinic counts and no-show counts from Python")
print(summary[["clinic_name", "appointments", "no_shows"]].sort_values("clinic_name"))
print("Total appointments:", summary["appointments"].sum())
print("Total no-shows:", summary["no_shows"].sum())

Clinic counts and no-show counts from Python
                clinic_name  appointments  no_shows
5  Edappally Medical Centre           142        23
7    Gomti Nagar Care Point            90        30
1      Hadapsar Care Centre           143        39
0     Kothrud Family Clinic           160        36
3      Malviya Nagar Clinic           142        38
2    Sitabuldi Health Point            82        13
4     Vaishali Nagar Clinic           130        18
6        Vijay Nagar Clinic           101        21
Total appointments: 990
Total no-shows: 218


In [19]:

#ML Model: Features, split and baseline


# SECTION 1: Import the tools and load the file

import pandas as pd
from sklearn.model_selection import train_test_split

# Do NOT sort or shuffle after loading: the row order matters for the split.
data = pd.read_csv("/content/appointments_clean.csv")

print("SECTION 1 - File loaded")
print("Shape:", data.shape)

# SECTION 2: Choose the feature columns

# Only these 8 columns are used (no patient_id, appointment_id, dates,
# clinic_id, gender or status).
feature_columns = ["age", "distance_km", "fee_inr", "lead_days",
                   "prior_no_shows", "reminder_sent",
                   "booking_channel", "visit_type"]

X = data[feature_columns].copy()

print("\nSECTION 2 - Feature columns:", list(X.columns))

# SECTION 3: Convert reminder_sent from Yes/No to 1/0

X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

print("\nSECTION 3 - reminder_sent values:")
print(X["reminder_sent"].value_counts())


# SECTION 4: One-hot encode booking_channel and visit_type

X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])

print("\nSECTION 4 - Columns after one-hot encoding:")
print(list(X.columns))
print("Shape of X:", X.shape)


# SECTION 5: Define the target y

y = data["no_show"]

print("\nSECTION 5 - Target no_show counts:")
print(y.value_counts())


# SECTION 6: Split into training and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("\nSECTION 6 - Split sizes")
print("Training size:", len(X_train))
print("Test size:", len(X_test))



# SECTION 7: No-shows in the test set

print("\nSECTION 7 - No-shows in the test set:", y_test.sum())

# SECTION 8: Baseline accuracy (always predict "Attended")

# "Attended" = no_show of 0, so the baseline is the share of 0s in the test set.
baseline_accuracy = (y_test == 0).mean()

print("\nSECTION 8 - Baseline accuracy (%):", round(baseline_accuracy * 100, 1))

SECTION 1 - File loaded
Shape: (990, 16)

SECTION 2 - Feature columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel', 'visit_type']

SECTION 3 - reminder_sent values:
reminder_sent
1    561
0    429
Name: count, dtype: int64

SECTION 4 - Columns after one-hot encoding:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Shape of X: (990, 11)

SECTION 5 - Target no_show counts:
no_show
0    772
1    218
Name: count, dtype: int64

SECTION 6 - Split sizes
Training size: 792
Test size: 198

SECTION 7 - No-shows in the test set: 49

SECTION 8 - Baseline accuracy (%): 75.3


In [20]:

 #Logistic regression

# Import the tools

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)


 # Scale the features (fit on the training set only)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # learns the mean and std from training data
X_test_s = scaler.transform(X_test)         # applies the SAME mean and std to test data

print("SECTION 2 - Scaled shapes:", X_train_s.shape, X_test_s.shape)

# Fit the logistic regression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

print("\nSECTION 3 - Model fitted")

#  Predict on the test set

y_pred = model.predict(X_test_s)                    # 0/1 predictions (threshold 0.5)
y_prob = model.predict_proba(X_test_s)[:, 1]        # probability of a no-show

#  Test-set metrics

print("\nSECTION 5 - Test-set metrics")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

print("\nConfusion matrix (rows = actual 0,1; columns = predicted 0,1):")
print(confusion_matrix(y_test, y_pred))

#  Coefficients next to the feature names

print("\nSECTION 6 - Coefficients (scaled features)")
for name, coef in zip(X.columns, model.coef_[0]):
    print(name, ":", round(coef, 4))

# Effect of reminder_sent

# Find the position of reminder_sent in the column order of X.
i = list(X.columns).index("reminder_sent")

coef_scaled = model.coef_[0][i]            # coefficient on the scaled feature
scale = scaler.scale_[i]                   # std of reminder_sent in the training set
coef_original = coef_scaled / scale        # coefficient per 1 unit (No -> Yes)

print("\nSECTION 7 - reminder_sent")
print("Coefficient (scaled):", round(coef_scaled, 4))
print("Std of reminder_sent (scaler.scale_):", round(scale, 4))
print("np.exp(scaled coefficient):", round(np.exp(coef_scaled), 3))
print("Coefficient / std:", round(coef_original, 4))
print("np.exp(coefficient / std), No -> Yes:", round(np.exp(coef_original), 3))

SECTION 2 - Scaled shapes: (792, 11) (198, 11)

SECTION 3 - Model fitted

SECTION 5 - Test-set metrics
Accuracy: 0.788
Precision: 0.63
Recall: 0.347
F1: 0.447
ROC-AUC: 0.789

Confusion matrix (rows = actual 0,1; columns = predicted 0,1):
[[139  10]
 [ 32  17]]

SECTION 6 - Coefficients (scaled features)
age : -0.2539
distance_km : 0.2534
fee_inr : 0.4528
lead_days : 0.6145
prior_no_shows : 0.6496
reminder_sent : -0.5807
booking_channel_App : -0.0468
booking_channel_Phone : 0.2159
booking_channel_Walk-In : -0.1864
visit_type_Consultation : 0.0157
visit_type_Follow-up : -0.0157

SECTION 7 - reminder_sent
Coefficient (scaled): -0.5807
Std of reminder_sent (scaler.scale_): 0.495
np.exp(scaled coefficient): 0.56
Coefficient / std: -1.1731
np.exp(coefficient / std), No -> Yes: 0.309


In [21]:

# TASK 3.3 - Decision tree and overfitting

# ------------------------------------------------------------
# SECTION 1: Import the tool
# ------------------------------------------------------------
from sklearn.tree import DecisionTreeClassifier


# ------------------------------------------------------------
# SECTION 2: Tree with no depth limit
# ------------------------------------------------------------
# With no limit, the tree keeps splitting until it fits the training data closely.
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

train_acc_full = tree_full.score(X_train, y_train)   # accuracy on training data
test_acc_full = tree_full.score(X_test, y_test)      # accuracy on test data

print("SECTION 2 - Tree with no depth limit")
print("Training accuracy:", round(train_acc_full, 3))
print("Test accuracy:", round(test_acc_full, 3))
print("Depth reached:", tree_full.get_depth())


# ------------------------------------------------------------
# SECTION 3: Tree with max_depth=3
# ------------------------------------------------------------
tree_small = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_small.fit(X_train, y_train)

train_acc_small = tree_small.score(X_train, y_train)
test_acc_small = tree_small.score(X_test, y_test)

print("\nSECTION 3 - Tree with max_depth=3")
print("Training accuracy:", round(train_acc_small, 3))
print("Test accuracy:", round(test_acc_small, 3))


# ------------------------------------------------------------
# SECTION 4: Gap between training and test accuracy
# ------------------------------------------------------------
print("\nSECTION 4 - Gap (training minus test)")
print("No depth limit:", round(train_acc_full - test_acc_full, 3))
print("max_depth=3:", round(train_acc_small - test_acc_small, 3))

SECTION 2 - Tree with no depth limit
Training accuracy: 1.0
Test accuracy: 0.672
Depth reached: 15

SECTION 3 - Tree with max_depth=3
Training accuracy: 0.822
Test accuracy: 0.778

SECTION 4 - Gap (training minus test)
No depth limit: 0.328
max_depth=3: 0.044


In [22]:
# ============================================================
# TASK 3.4 - Random forest and a comparison table
# ============================================================


# ------------------------------------------------------------
# SECTION 1: Import the tools
# ------------------------------------------------------------
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score


# ------------------------------------------------------------
# SECTION 2: Fit the random forest
# ------------------------------------------------------------
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)                  # 0/1 predictions
rf_prob = rf.predict_proba(X_test)[:, 1]      # probability of a no-show

print("SECTION 2 - Random forest test-set metrics")
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 3))
print("Recall:", round(recall_score(y_test, rf_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 3))


# ------------------------------------------------------------
# SECTION 3: Top three feature importances
# ------------------------------------------------------------
importances = pd.Series(rf.feature_importances_, index=X.columns)
top_three = importances.sort_values(ascending=False).head(3)

print("\nSECTION 3 - Top three feature importances")
print(top_three.round(3))


# ------------------------------------------------------------
# SECTION 4: Predictions for the other two models
# ------------------------------------------------------------
# Logistic regression (uses the scaled test features)
lr_pred = model.predict(X_test_s)
lr_prob = model.predict_proba(X_test_s)[:, 1]

# Depth-3 decision tree (uses the unscaled test features)
tree_pred = tree_small.predict(X_test)
tree_prob = tree_small.predict_proba(X_test)[:, 1]


# ------------------------------------------------------------
# SECTION 5: Comparison table
# ------------------------------------------------------------
comparison = pd.DataFrame({
    "Model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "Accuracy": [accuracy_score(y_test, lr_pred),
                 accuracy_score(y_test, tree_pred),
                 accuracy_score(y_test, rf_pred)],
    "Recall": [recall_score(y_test, lr_pred),
               recall_score(y_test, tree_pred),
               recall_score(y_test, rf_pred)],
    "ROC-AUC": [roc_auc_score(y_test, lr_prob),
                roc_auc_score(y_test, tree_prob),
                roc_auc_score(y_test, rf_prob)],
}).round(3)

print("\nSECTION 5 - Comparison table (test set, for reporting only)")
print(comparison.to_string(index=False))

SECTION 2 - Random forest test-set metrics
Accuracy: 0.747
Recall: 0.306
ROC-AUC: 0.706

SECTION 3 - Top three feature importances
lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64

SECTION 5 - Comparison table (test set, for reporting only)
                  Model  Accuracy  Recall  ROC-AUC
    Logistic regression     0.788   0.347    0.789
Decision tree (depth 3)     0.778   0.224    0.700
          Random forest     0.747   0.306    0.706


In [23]:
# ============================================================
# TASK 3.5 - Stage 1: parts (a) and (b), training set only
# ============================================================


# ------------------------------------------------------------
# SECTION 1: Import the tools
# ------------------------------------------------------------
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# SECTION 2: (a) Tune the random forest with GridSearchCV
# ------------------------------------------------------------
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)

print("SECTION 2 - (a) Grid search")
print("best_params_:", grid.best_params_)
print("best_score_ (cross-validated ROC-AUC):", round(grid.best_score_, 3))


# ------------------------------------------------------------
# SECTION 3: (b) Cross-validated ROC-AUC of each candidate
# ------------------------------------------------------------
# Logistic regression: the pipeline re-fits the scaler inside every fold.
lr_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))])

tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
rf_default = RandomForestClassifier(n_estimators=100, random_state=42)

cv_scores = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(tree3, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (untuned)": cross_val_score(rf_default, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (tuned, best_score_)": grid.best_score_,
}

print("\nSECTION 3 - (b) 5-fold cross-validated ROC-AUC on the training set")
for name, score in cv_scores.items():
    print(name, ":", round(score, 3))


# ------------------------------------------------------------
# SECTION 4: (b) The winner
# ------------------------------------------------------------
winner = max(cv_scores, key=cv_scores.get)

print("\nSECTION 4 - Winner (highest cross-validated ROC-AUC):", winner)

SECTION 2 - (a) Grid search
best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (cross-validated ROC-AUC): 0.767

SECTION 3 - (b) 5-fold cross-validated ROC-AUC on the training set
Logistic regression (pipeline) : 0.774
Decision tree (depth 3) : 0.678
Random forest (untuned) : 0.747
Random forest (tuned, best_score_) : 0.767

SECTION 4 - Winner (highest cross-validated ROC-AUC): Logistic regression (pipeline)


In [24]:
# ============================================================
# TASK 3.5 - Stage 2: parts (c), (d) and (e)
# ============================================================


# ------------------------------------------------------------
# SECTION 1: Import the tools
# ------------------------------------------------------------
import pandas as pd
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


# ------------------------------------------------------------
# SECTION 2: (c) Out-of-fold training probabilities
# ------------------------------------------------------------
# Each training row gets a probability from a model that did NOT see that row.
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("SECTION 2 - (c) Out-of-fold probabilities calculated:", len(oof))


# ------------------------------------------------------------
# SECTION 3: (c) Compare thresholds 0.5 and 0.35
# ------------------------------------------------------------
results = []
for threshold in [0.5, 0.35]:
    flagged = (oof >= threshold).astype(int)      # 1 = flagged as likely no-show
    results.append({
        "threshold": threshold,
        "precision": round(precision_score(y_train, flagged), 3),
        "recall": round(recall_score(y_train, flagged), 3),
        "F1": round(f1_score(y_train, flagged), 3),
        "number_flagged": int(flagged.sum()),
    })

threshold_table = pd.DataFrame(results)

print("\nSECTION 3 - (c) Thresholds on out-of-fold training predictions")
print(threshold_table.to_string(index=False))


# ------------------------------------------------------------
# SECTION 4: (c) Pick the threshold with the higher F1
# ------------------------------------------------------------
best_row = threshold_table.loc[threshold_table["F1"].idxmax()]
chosen_threshold = float(best_row["threshold"])

print("\nSECTION 4 - (c) Chosen threshold (higher F1):", chosen_threshold)


# ------------------------------------------------------------
# SECTION 5: (d) Fit the winner and evaluate once on the test set
# ------------------------------------------------------------
lr_pipe.fit(X_train, y_train)                         # the pipeline fits its own scaler
probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flag = (probs >= chosen_threshold).astype(int)

print("\nSECTION 5 - (d) Test set at threshold", chosen_threshold)
print("Precision:", round(precision_score(y_test, test_flag), 3))
print("Recall:", round(recall_score(y_test, test_flag), 3))
print("Number flagged:", int(test_flag.sum()), "of", len(y_test))
print("ROC-AUC:", round(roc_auc_score(y_test, probs), 3))


# ------------------------------------------------------------
# SECTION 6: (e) Save test_risk_scores.csv
# ------------------------------------------------------------
# X_test.index gives the original row numbers, so we can look up appointment_id.
scores = pd.DataFrame({
    "appointment_id": data.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": test_flag,
})

scores.to_csv("test_risk_scores.csv", index=False)

print("\nSECTION 6 - (e) Saved test_risk_scores.csv")
print("Rows:", len(scores))
print("Columns:", list(scores.columns))
print(scores.head())

SECTION 2 - (c) Out-of-fold probabilities calculated: 792

SECTION 3 - (c) Thresholds on out-of-fold training predictions
 threshold  precision  recall    F1  number_flagged
      0.50      0.575   0.249 0.347              73
      0.35      0.513   0.467 0.489             154

SECTION 4 - (c) Chosen threshold (higher F1): 0.35

SECTION 5 - (d) Test set at threshold 0.35
Precision: 0.571
Recall: 0.571
Number flagged: 49 of 198
ROC-AUC: 0.789

SECTION 6 - (e) Saved test_risk_scores.csv
Rows: 198
Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0


In [ ]:
# ============================================================
# TASK 4.1 - Cell 1: install, key and client
# ============================================================

# ------------------------------------------------------------
# SECTION 1: Install the libraries
# ------------------------------------------------------------
!pip install -q google-genai pydantic

# ------------------------------------------------------------
# SECTION 2: Import the tools
# ------------------------------------------------------------
import json
import pandas as pd
from google import genai
from google.genai import types

# ------------------------------------------------------------
# SECTION 3: Type the key and model name at run time
# ------------------------------------------------------------
# input() asks you to type it. The key is never written in the code.
API_KEY = input("Paste your key: ")
MODEL_NAME = input("Model name (without 'models/'): ")

client = genai.Client(api_key=API_KEY)
print("Client ready for model:", MODEL_NAME)

In [30]:
# ============================================================
# TASK 4.1 - Cell 2: analyse_note and the run on N001
# ============================================================

# ------------------------------------------------------------
# SECTION 1: The JSON schema (four fields)
# ------------------------------------------------------------
# root_cause is a plain string: the allowed values are checked later, in Task 4.2.
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

# ------------------------------------------------------------
# SECTION 2: The system instruction (the Role of CRAFT)
# ------------------------------------------------------------
SYSTEM_TEXT = ("You are a careful analyst for an outpatient clinic chain. "
               "You read short call-centre notes and classify why a patient missed an appointment.")

# ------------------------------------------------------------
# SECTION 3: The function
# ------------------------------------------------------------
def analyse_note(note_text):

    # The prompt uses the other CRAFT parts: Context, Action, Tone, Format.
    prompt = f"""
CONTEXT: CareLoop Clinics runs eight outpatient clinics in India. A call-centre agent phoned a patient who missed an appointment and wrote the note below.

ACTION: Read the note and decide the main reason the patient missed the appointment. Choose root_cause from exactly these values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. Give a confidence between 0 and 1. Write a summary of at most 25 words. Suggest one recommended_action for the clinic.

TONE: Neutral, factual and brief.

FORMAT: Return only JSON with the fields root_cause, confidence, summary and recommended_action.

NOTE:
{note_text}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=2000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))

    # Turn the JSON text into a Python dictionary.
    return json.loads(response.text)

# ------------------------------------------------------------
# SECTION 4: Load the notes and find N001
# ------------------------------------------------------------
notes = pd.read_csv("/content/followup_notes.csv")
note_n001 = notes[notes["note_id"] == "N001"].iloc[0]

print("SECTION 4 - Note N001")
print(note_n001["note_text"])

# ------------------------------------------------------------
# SECTION 5: Run the function and print the dictionary
# ------------------------------------------------------------
result = analyse_note(note_n001["note_text"])

print("\nSECTION 5 - Parsed dictionary")
print(result)

SECTION 4 - Note N001
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

SECTION 5 - Parsed dictionary
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient forgot appointment because she did not receive a reminder message and requested to reschedule for next week.', 'recommended_action': 'Verify patient contact details and audit automated SMS/messaging reminder system for delivery failures.'}


In [31]:
# ============================================================
# TASK 4.2 - Cell A: raw-reply function, Pydantic model, allowed list
# ============================================================

# ------------------------------------------------------------
# SECTION 1: Import the tools
# ------------------------------------------------------------
import json
import time
import pandas as pd
from google.genai import types
from pydantic import BaseModel, Field

# ------------------------------------------------------------
# SECTION 2: The schema and system text (same as Task 4.1)
# ------------------------------------------------------------
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You are a careful analyst for an outpatient clinic chain. "
               "You read short call-centre notes and classify why a patient missed an appointment.")

# ------------------------------------------------------------
# SECTION 3: Function that returns the model's reply text, unchanged
# ------------------------------------------------------------
def get_raw_reply(note_text):

    prompt = f"""
CONTEXT: CareLoop Clinics runs eight outpatient clinics in India. A call-centre agent phoned a patient who missed an appointment and wrote the note below.

ACTION: Read the note and decide the main reason the patient missed the appointment. Choose root_cause from exactly these values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. Give a confidence between 0 and 1. Write a summary of at most 25 words. Suggest one recommended_action for the clinic.

TONE: Neutral, factual and brief.

FORMAT: Return only JSON with the fields root_cause, confidence, summary and recommended_action.

NOTE:
{note_text}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=2000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))

    # Return the text exactly as the model sent it.
    return response.text

# ------------------------------------------------------------
# SECTION 4: analyse_note behaves as in Task 4.1
# ------------------------------------------------------------
def analyse_note(note_text):
    return json.loads(get_raw_reply(note_text))

# ------------------------------------------------------------
# SECTION 5: (a) The Pydantic model
# ------------------------------------------------------------
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# ------------------------------------------------------------
# SECTION 6: (a) The allowed-list check
# ------------------------------------------------------------
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

def root_cause_is_allowed(value):
    return value in ALLOWED

print("Cell A ready. Allowed root causes:", ALLOWED)

Cell A ready. Allowed root causes: ['forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule', 'other']


In [32]:
# ============================================================
# TASK 4.2 - Cell B: process the first 10 notes
# ============================================================

# ------------------------------------------------------------
# SECTION 1: Load the notes
# ------------------------------------------------------------
notes = pd.read_csv("/content/followup_notes.csv")

four_fields = ["root_cause", "confidence", "summary", "recommended_action"]
records = []          # one dictionary per note will be collected here

print("SECTION 1 - Notes loaded:", len(notes), "| processing the first 10")

# ------------------------------------------------------------
# SECTION 2: (b) Loop over the first 10 rows
# ------------------------------------------------------------
print("\nSECTION 2 - Processing")

for index, note_row in notes.head(10).iterrows():

    note_text = note_row["note_text"]

    # Start with an empty record for this note.
    record = {"note_id": note_row["note_id"],
              "appointment_id": note_row["appointment_id"],
              "root_cause": None, "confidence": None,
              "summary": None, "recommended_action": None,
              "validation_status": None, "raw_response": None}

    # ---- Blank note: do not call the API ----
    # pd.isna is tested first, because a blank cell is NaN (a float).
    if pd.isna(note_text) or str(note_text).strip() == "":
        record["root_cause"] = "other"
        record["confidence"] = 0.0
        record["summary"] = "No note recorded"
        record["recommended_action"] = "Re-call patient"
        record["validation_status"] = "skipped_blank"
        record["raw_response"] = ""

    # ---- Real note: call the API and validate ----
    else:
        try:
            raw = get_raw_reply(note_text)
            record["raw_response"] = raw            # keep the reply exactly as returned

            data = json.loads(raw)

            # Copy whichever of the four fields the reply contained.
            if isinstance(data, dict):
                for field in four_fields:
                    record[field] = data.get(field)

            # Pydantic check, then the allowed-list check.
            FollowupInsight(**data)
            if not root_cause_is_allowed(data["root_cause"]):
                raise ValueError("root_cause is not in the allowed list")

            record["validation_status"] = "valid"

        except Exception as error:
            # One bad reply must never stop the loop.
            record["validation_status"] = "invalid"
            print("   ", record["note_id"], "problem:", error)

        # (c) Pause between calls (free-tier rate limit).
        time.sleep(13)

    records.append(record)
    print(record["note_id"], "->", record["validation_status"], "|", record["root_cause"])

# ------------------------------------------------------------
# SECTION 3: (d) Save notes_analysed.csv
# ------------------------------------------------------------
columns = ["note_id", "appointment_id", "root_cause", "confidence", "summary",
           "recommended_action", "validation_status", "raw_response"]

notes_analysed = pd.DataFrame(records)[columns]
notes_analysed.to_csv("notes_analysed.csv", index=False)

print("\nSECTION 3 - Saved notes_analysed.csv with", len(notes_analysed), "rows")

# ------------------------------------------------------------
# SECTION 4: Summary counts
# ------------------------------------------------------------
print("\nSECTION 4 - validation_status counts")
print(notes_analysed["validation_status"].value_counts())

print("\nroot_cause counts")
print(notes_analysed["root_cause"].value_counts())

print("\nPreview")
print(notes_analysed[["note_id", "root_cause", "confidence", "validation_status"]])

SECTION 1 - Notes loaded: 30 | processing the first 10

SECTION 2 - Processing
N001 -> valid | forgot_or_no_reminder
N002 -> valid | distance_or_transport
N003 -> valid | cost_or_fee
N004 -> skipped_blank | other
N005 -> valid | illness_or_emergency
N006 -> valid | work_or_schedule
N007 -> valid | forgot_or_no_reminder
N008 -> valid | distance_or_transport
N009 -> valid | other
N010 -> valid | work_or_schedule

SECTION 3 - Saved notes_analysed.csv with 10 rows

SECTION 4 - validation_status counts
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

root_cause counts
root_cause
forgot_or_no_reminder    2
distance_or_transport    2
other                    2
work_or_schedule         2
cost_or_fee              1
illness_or_emergency     1
Name: count, dtype: int64

Preview
  note_id             root_cause  confidence validation_status
0    N001  forgot_or_no_reminder        0.98             valid
1    N002  distance_or_transport        0.95             valid

In [33]:
# ============================================================
# TASK 4.3 - Break it on purpose
# ============================================================


# ------------------------------------------------------------
# SECTION 1: Import the tools
# ------------------------------------------------------------
import json
import time
import pandas as pd
from google.genai import types
from pydantic import ValidationError


# ------------------------------------------------------------
# SECTION 2: (a) Two deliberately bad replies
# ------------------------------------------------------------
# Bad reply 1: confidence is 1.7, but the model only allows 0 to 1.
bad_reply_1 = {"root_cause": "other", "confidence": 1.7,
               "summary": "Patient did not come", "recommended_action": "Call the patient"}

# Bad reply 2: recommended_action is missing.
bad_reply_2 = {"root_cause": "other", "confidence": 0.5,
               "summary": "Patient did not come"}

print("SECTION 2 - (a) Validation errors")

try:
    FollowupInsight(**bad_reply_1)
except ValidationError as error:
    print("\nBad reply 1 (confidence = 1.7):")
    print(error)

try:
    FollowupInsight(**bad_reply_2)
except ValidationError as error:
    print("\nBad reply 2 (recommended_action missing):")
    print(error)


# ------------------------------------------------------------
# SECTION 3: (b) A function that calls the API with ANY prompt
# ------------------------------------------------------------
# Only the prompt changes. Schema, system text and settings are the same.
api_calls_here = 0      # counts the calls made in this task

def call_with_prompt(prompt):
    global api_calls_here
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=2000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))
    api_calls_here = api_calls_here + 1
    return response.text


# ------------------------------------------------------------
# SECTION 4: (b) The two prompts
# ------------------------------------------------------------
def prompt_v1(note_text):
    return f"Classify this note:\n{note_text}"

def prompt_v2(note_text):
    return f"""
CONTEXT: CareLoop Clinics runs eight outpatient clinics in India. A call-centre agent phoned a patient who missed an appointment and wrote the note below.

ACTION: Read the note and decide the main reason the patient missed the appointment. Choose root_cause from exactly these values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. If the note is ambiguous or no listed reason clearly fits, answer other rather than guess. Give a confidence between 0 and 1. Write a summary of at most 25 words. Suggest one recommended_action for the clinic.

TONE: Neutral, factual and brief.

FORMAT: Return only JSON with the fields root_cause, confidence, summary and recommended_action.

NOTE:
{note_text}
"""


# ------------------------------------------------------------
# SECTION 5: (b) Run both prompts on N007 and N009
# ------------------------------------------------------------
rows = []

for note_id in ["N007", "N009"]:

    note_text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]
    print("\n" + note_id + " note text:", note_text)

    for version, make_prompt in [("V1", prompt_v1), ("V2", prompt_v2)]:

        try:
            raw = call_with_prompt(make_prompt(note_text))
            data = json.loads(raw)
            root_cause = data.get("root_cause")
            confidence = data.get("confidence")
        except Exception as error:
            root_cause = "ERROR: " + str(error)[:60]
            confidence = None

        rows.append({"note_id": note_id,
                     "prompt": version,
                     "root_cause": root_cause,
                     "confidence": confidence,
                     "in_allowed_list": root_cause in ALLOWED})

        time.sleep(13)      # free-tier rate limit


# ------------------------------------------------------------
# SECTION 6: (b) The four results in a small table
# ------------------------------------------------------------
print("\nSECTION 6 - (b) Prompt comparison")
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


# ------------------------------------------------------------
# SECTION 7: (c) Total API calls across Tasks 4.1 to 4.3
# ------------------------------------------------------------
calls_4_1 = 1                                                        # the N001 test
calls_4_2 = int((notes_analysed["validation_status"] != "skipped_blank").sum())   # real notes only
calls_4_3 = api_calls_here

print("\nSECTION 7 - (c) API calls used")
print("Task 4.1:", calls_4_1)
print("Task 4.2:", calls_4_2)
print("Task 4.3:", calls_4_3)
print("Total:", calls_4_1 + calls_4_2 + calls_4_3)

SECTION 2 - (a) Validation errors

Bad reply 1 (confidence = 1.7):
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Bad reply 2 (recommended_action missing):
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '... 'Patient did not come'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

N007 note text: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.

N009 note text: Phone was switched off, could not reach the patient after three attempts.

SECTION 6 - (b) Prompt comparison
note_id prompt                               root_cause  confidence  in_allowed_list
   N007     V1 Forgotten appointment / Lack of reminder        0.95            False
   N007     V2                    f

In [34]:
# ============================================================
# TASK 4.4 (a) - Merge notes with appointments and cross-tabulate
# ============================================================

# ------------------------------------------------------------
# SECTION 1: Load both files
# ------------------------------------------------------------
import pandas as pd

notes_analysed = pd.read_csv("/content/notes_analysed.csv")
appointments = pd.read_csv("/content/appointments_clean.csv")

print("SECTION 1 - Notes rows:", len(notes_analysed), "| Appointments rows:", len(appointments))

# ------------------------------------------------------------
# SECTION 2: Merge on appointment_id
# ------------------------------------------------------------
# An inner merge keeps only notes whose appointment_id exists in appointments.
merged = notes_analysed.merge(appointments, on="appointment_id", how="inner")

print("\nSECTION 2 - Rows after merge:", len(merged))
print(merged[["note_id", "appointment_id", "root_cause", "reminder_sent", "status"]])

# ------------------------------------------------------------
# SECTION 3: Cross-tabulate root_cause against reminder_sent
# ------------------------------------------------------------
# margins=True adds the totals row and column.
table = pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True)

print("\nSECTION 3 - root_cause vs reminder_sent")
print(table)

SECTION 1 - Notes rows: 10 | Appointments rows: 990

SECTION 2 - Rows after merge: 10
  note_id appointment_id             root_cause reminder_sent   status
0    N001          A0872  forgot_or_no_reminder            No  No-Show
1    N002          A0897  distance_or_transport            No  No-Show
2    N003          A0526            cost_or_fee           Yes  No-Show
3    N004          A0926                  other            No  No-Show
4    N005          A0534   illness_or_emergency           Yes  No-Show
5    N006          A0507       work_or_schedule           Yes  No-Show
6    N007          A0333  forgot_or_no_reminder            No  No-Show
7    N008          A0222  distance_or_transport           Yes  No-Show
8    N009          A0977                  other           Yes  No-Show
9    N010          A0596       work_or_schedule            No  No-Show

SECTION 3 - root_cause vs reminder_sent
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee         